In [ ]:
# @title Install required dependencies
!pip install -q "crewai[tools]>=1.8.0" python-dotenv ddgs pydantic pandas pypdf


In [ ]:
import os
from pydantic import BaseModel
from google.colab import userdata
from crewai import Agent, Task, Crew, LLM
from crewai.flow.flow import Flow, listen, or_, router, start

class BugTriageState(BaseModel):
    issue_title: str = ""
    issue_body: str = ""
    issue_type: str = ""
    resolution: str = ""

try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    llm = LLM(
        model="google/gemini-3.1-flash-lite-preview",
        api_key=userdata.get("GEMINI_API_KEY"),
        temperature=0.2,
        max_tokens=1000,
    )

    class BugTriageFlow(Flow[BugTriageState]):
        @start()
        def receive_issue(self):
            title = input("Enter issue title [or press Enter for default]: ").strip()
            if not title:
                title = "App crashes with 500 status on checkout when using PayPal"
            body = input("Enter issue details [or press Enter for default]: ").strip()
            if not body:
                body = "Traceback indicates KeyError 'order_id' in payments/paypal.py line 42 on Safari mobile."
            self.state.issue_title = title
            self.state.issue_body = body

        @listen(receive_issue)
        def classify_issue(self):
            prompt = (
                f"Classify this GitHub issue as exactly one word: bug, feature, or question.\n"
                f"Title: {self.state.issue_title}\n"
                f"Body: {self.state.issue_body}\n"
                f"Reply with ONLY the lowercase classification word."
            )
            cat = llm.call(prompt).strip().lower()
            if "feature" in cat:
                self.state.issue_type = "feature"
            elif "question" in cat:
                self.state.issue_type = "question"
            else:
                self.state.issue_type = "bug"

        @router(classify_issue)
        def route_issue(self):
            return self.state.issue_type

        @listen("bug")
        def handle_bug(self):
            engineer = Agent(
                role="QA Triager",
                goal="Assess bug severity, estimate blast radius, and structure reproduction steps",
                backstory="Reliability engineer diagnosing defects and triaging critical fixes.",
                llm=llm,
                verbose=False,
            )
            task = Task(
                description=f"Analyze defect: {self.state.issue_title}. Details: {self.state.issue_body}. Format as Severity, Steps to Reproduce, and Root Cause Hypothesis.",
                expected_output="Formatted bug triage memo.",
                agent=engineer,
            )
            self.state.resolution = Crew(agents=[engineer], tasks=[task]).kickoff().raw

        @listen("feature")
        def handle_feature(self):
            pm = Agent(
                role="Product Manager",
                goal="Convert feature request into a user story with acceptance criteria",
                backstory="Technical PM defining user stories and product requirements.",
                llm=llm,
                verbose=False,
            )
            task = Task(
                description=f"Formulate feature spec for: {self.state.issue_title}. Context: {self.state.issue_body}.",
                expected_output="User story with Given/When/Then acceptance criteria.",
                agent=pm,
            )
            self.state.resolution = Crew(agents=[pm], tasks=[task]).kickoff().raw

        @listen("question")
        def handle_question(self):
            prompt = f"Answer this developer support question clearly and provide code or docs references where applicable:\n{self.state.issue_title}\n{self.state.issue_body}"
            self.state.resolution = llm.call(prompt)

    flow = BugTriageFlow()
    flow.kickoff()
    print(f"\n--- Triage Result: [{flow.state.issue_type.upper()}] ---")
    print(flow.state.resolution)

except Exception as e:
    print("Error:", e)
